# Visuelle 2.0 preprocessing

**Owner:** Nilakshi  
**Module:** CCS4310 – Deep Learning  

Create a leakage-safe chronological demand table from the locally available Visuelle `sales.csv` data.

In [18]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
RANDOM_STATE=42
FAST_DEV_RUN=True
FORECAST_MODE='short_observation'  # supported: short_observation, cold_start
assert FORECAST_MODE in {'short_observation','cold_start'}
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'data'/'raw'/'visuelle').is_dir(): return p
    raise FileNotFoundError('Repository root not found.')
ROOT=find_root(); RAW=ROOT/'data'/'raw'/'visuelle'; OUT=ROOT/'data'/'interim'; OUT.mkdir(parents=True,exist_ok=True)
assert (RAW/'sales.csv').is_file()

## Load and reshape actual source columns

In [19]:
sales=pd.read_csv(RAW/'sales.csv')
weekly_cols=[c for c in sales.columns if str(c).isdigit()]
assert weekly_cols and 'release_date' in sales.columns
id_cols=[c for c in ['external_code','retail'] if c in sales.columns]
long=sales.melt(id_vars=[c for c in sales.columns if c not in weekly_cols],value_vars=weekly_cols,var_name='period_index',value_name='demand')
long['period_index']=pd.to_numeric(long['period_index'],errors='coerce')
long['release_date']=pd.to_datetime(long['release_date'],errors='coerce')
long['time']=long['release_date']+pd.to_timedelta(long['period_index'],unit='W')
long['demand']=pd.to_numeric(long['demand'],errors='coerce')
rows_before=len(long); duplicates_before=int(long.duplicated().sum())
long=long.dropna(subset=['time','demand']).drop_duplicates().sort_values(id_cols+['time']).reset_index(drop=True)
print('Weekly interpretation: period_index is weeks after release_date.')
print('rows:',len(long),'removed:',rows_before-len(long),'date range:',long.time.min(),'to',long.time.max())

Weekly interpretation: period_index is weeks after release_date.
rows: 1282200 removed: 0 date range: 2016-11-28 00:00:00 to 2020-03-16 00:00:00


## Clean values and generate past-only features

**Restock policy:** `sales.csv.restock` is a single product/shop field with no weekly timestamp, so it is treated as aggregate lifecycle information and excluded rather than made safe by shifting rows. The usable source is `restocks.csv`, whose actual keys are `external_code`, `retail`, `week`, `year`, and `qty`. For `short_observation`, only an exact calendar-week match shifted by one row within product/shop is allowed. For `cold_start`, all direct product restock/stock features are excluded because they are not established as pre-launch information.

In [20]:
for c in ['demand']:
    if c in long:
        long[c]=pd.to_numeric(long[c],errors='coerce'); long.loc[long[c]<0,c]=np.nan
rows_after_invalid=len(long); long=long.dropna(subset=['time','demand']).copy()
for c in ['category','color','fabric','season']:
    if c in long: long[c]=long[c].fillna('Unknown').astype(str)
long['year']=long.time.dt.year; long['month']=long.time.dt.month; long['calendar_week']=long.time.dt.isocalendar().week.astype('Int64'); long['day_of_week']=long.time.dt.dayofweek
long['season_from_time']=long.month.map({12:'winter',1:'winter',2:'winter',3:'spring',4:'spring',5:'spring',6:'summer',7:'summer',8:'summer',9:'autumn',10:'autumn',11:'autumn'}).fillna('Unknown')
if (RAW/'price_discount_series.csv').is_file() and set(id_cols).issubset({'external_code','retail'}):
    pdx=pd.read_csv(RAW/'price_discount_series.csv')
    period_price=pdx.melt(id_vars=id_cols+['price'],value_vars=[c for c in weekly_cols if c in pdx.columns],var_name='period_index',value_name='discount')
    period_price['period_index']=pd.to_numeric(period_price['period_index'],errors='coerce')
    long=long.merge(period_price,on=id_cols+['period_index'],how='left',validate='many_to_one')
    pre=period_price[period_price.period_index==0][id_cols+['price','discount']].rename(columns={'price':'price_prelaunch','discount':'discount_prelaunch'}).drop_duplicates(id_cols)
    long=long.merge(pre,on=id_cols,how='left',validate='many_to_one')
    print('Exact price/discount matches:',int(long.discount.notna().sum()),'of',len(long))
else:
    print('Price/discount excluded: reliable product/shop/weekly keys unavailable.')
restock_joined=False
if (RAW/'restocks.csv').is_file() and set(id_cols+['year','calendar_week']).issubset(long.columns):
    restocks=pd.read_csv(RAW/'restocks.csv')
    if {'external_code','retail','year','week','qty'}.issubset(restocks.columns):
        restocks=restocks.rename(columns={'week':'calendar_week','qty':'restock_weekly'})
        restocks['calendar_week']=pd.to_numeric(restocks['calendar_week'],errors='coerce').astype('Int64'); restocks['year']=pd.to_numeric(restocks['year'],errors='coerce').astype('Int64')
        restocks=restocks[['external_code','retail','year','calendar_week','restock_weekly']].dropna(subset=['external_code','retail','year','calendar_week'])
        restocks=restocks.groupby(['external_code','retail','year','calendar_week'],as_index=False)['restock_weekly'].sum()
        long=long.merge(restocks,on=id_cols+['year','calendar_week'],how='left',validate='many_to_one'); restock_joined=True
        print('Exact weekly restock matches after same-week aggregation:',int(long.restock_weekly.notna().sum()),'of',len(long))
# Shift all historical signals after sorting by product/shop and time.
long=long.sort_values(id_cols+['time']).reset_index(drop=True); group_cols=id_cols or ['category']; grouped=long.groupby(group_cols,sort=False,dropna=False)
long['demand_lag_1']=grouped.demand.shift(1); long['demand_lag_3']=grouped.demand.shift(3)
long['demand_rolling_mean_3']=grouped.demand.transform(lambda s:s.shift(1).rolling(3,min_periods=1).mean()); long['demand_rolling_std_3']=grouped.demand.transform(lambda s:s.shift(1).rolling(3,min_periods=2).std())
if restock_joined: long['restock_past_1']=grouped.restock_weekly.shift(1)
if 'price' in long: long['price_past_1']=grouped.price.shift(1)
if 'discount' in long: long['discount_past_1']=grouped.discount.shift(1)
for c in ['demand_rolling_mean_3','demand_rolling_std_3']: long[c]=long[c].fillna(0)
if 'retail' in long: long['shop_label']=long['retail'].astype('string').fillna('Unknown')
print('past-only features:',[c for c in long if 'lag' in c or 'rolling' in c or c.endswith('_past_1')])

Exact price/discount matches: 1278813 of 1278814
Exact weekly restock matches after same-week aggregation: 311145 of 1278814
past-only features: ['demand_lag_1', 'demand_lag_3', 'demand_rolling_mean_3', 'demand_rolling_std_3', 'restock_past_1', 'price_past_1', 'discount_past_1']


## Chronological train/validation/test split

In [21]:
long=long.sort_values('time').reset_index(drop=True)
dates=np.array(sorted(long.time.dropna().unique())); assert len(dates)>=3
i1=max(1,int(len(dates)*.70)); i2=min(len(dates)-1,max(i1+1,int(len(dates)*.85)))
bounds=[dates[0],dates[i1],dates[i2],dates[-1]]
train=long[long.time<bounds[1]].copy(); valid=long[(long.time>=bounds[1])&(long.time<bounds[2])].copy(); test=long[long.time>=bounds[2]].copy()
# Cold-start additionally requires unseen products in later splits where the data permits it.
if FORECAST_MODE=='cold_start' and 'external_code' in long:
    train_products=set(train.external_code.dropna()); valid=valid[~valid.external_code.isin(train_products)].copy()
    valid_products=set(valid.external_code.dropna()); test=test[~test.external_code.isin(train_products|valid_products)].copy()
assert train.time.max()<valid.time.min()<test.time.min()
for name,df in [('train',train),('validation',valid),('test',test)]:
    print(name,'rows=',len(df),'range=',df.time.min(),'to',df.time.max(),'unique_products=',df.external_code.nunique() if 'external_code' in df else None)
print('product overlap train/validation:',len(set(train.external_code)&set(valid.external_code)) if 'external_code' in long else 'n/a')
print('product overlap validation/test:',len(set(valid.external_code)&set(test.external_code)) if 'external_code' in long else 'n/a')
assert len(valid)>0 and len(test)>0

train rows= 794042 range= 2016-11-28 00:00:00 to 2019-03-18 00:00:00 unique_products= 3916
validation rows= 262417 range= 2019-03-25 00:00:00 to 2019-09-16 00:00:00 unique_products= 1407
test rows= 222355 range= 2019-09-23 00:00:00 to 2020-03-16 00:00:00 unique_products= 1045
product overlap train/validation: 483
product overlap validation/test: 530


## Feature configuration and persisted artifacts

In [22]:
target='demand'; time_col='time'
identifier_columns=id_cols
# Real product season and calendar season are distinct: both are retained intentionally.
categorical_candidates=[c for c in ['season','season_from_time','category','color','fabric','shop_label'] if c in long.columns]
time_features=[c for c in ['year','month','calendar_week','day_of_week','season_from_time'] if c in long.columns]
lag_features=[c for c in ['demand_lag_1','demand_lag_3','demand_rolling_mean_3','demand_rolling_std_3','restock_past_1','price_past_1','discount_past_1'] if c in long.columns]
if FORECAST_MODE=='short_observation':
    allowed=categorical_candidates+time_features+lag_features
else:
    allowed=categorical_candidates+time_features+[c for c in ['price_prelaunch','discount_prelaunch'] if c in long.columns]
    lag_features=[]
features=[]
for c in allowed:
    if c not in identifier_columns and c in long.columns and c not in features: features.append(c)
numerical_features=[c for c in features if pd.api.types.is_numeric_dtype(long[c]) and c not in categorical_candidates]
categorical_features=[c for c in features if c in categorical_candidates]
excluded_leakage_features=['demand','future demand','external_code/raw product ID','sales.csv.restock aggregate lifecycle field','future/post-launch restock','current/future price/discount','customer outcomes']
if FORECAST_MODE=='cold_start': excluded_leakage_features.append('restock_weekly and all direct product stock/restock fields')
restock_policy={'sales.csv.restock':'excluded: no weekly timestamp; treated as aggregate lifecycle information','restocks.csv.restock_weekly':('short_observation: exact calendar-week join then groupby product/shop shift(1); cold_start: excluded'),'current/future restock':'never allowed'}
cfg={'forecast_mode':FORECAST_MODE,'target_column':target,'time_column':time_col,'time_unit':'week','identifier_columns':identifier_columns,'numerical_features':numerical_features,'categorical_features':categorical_features,'time_features':time_features,'lag_features':lag_features,'feature_columns':features,'excluded_leakage_features':excluded_leakage_features,'leakage_exclusions':excluded_leakage_features,'restock_policy':restock_policy,'source_file':'data/raw/visuelle/sales.csv','random_state':RANDOM_STATE}
for name,df in [('full',long),('train',train),('validation',valid),('test',test)]: df.to_csv(OUT/f'visuelle_demand_{name}.csv',index=False)
(OUT/'visuelle_demand_feature_config.json').write_text(json.dumps(cfg,indent=2,default=str),encoding='utf-8')
summary={'forecast_mode':FORECAST_MODE,'target':target,'time_unit':'week','time_ranges':{k:[str(v.time.min()),str(v.time.max())] for k,v in [('train',train),('validation',valid),('test',test)]},'row_counts':{k:len(v) for k,v in [('full',long),('train',train),('validation',valid),('test',test)]},'unique_products':{k:int(v.external_code.nunique()) for k,v in [('train',train),('validation',valid),('test',test)] if 'external_code' in v},'feature_groups':{'categorical':categorical_features,'numerical':numerical_features,'time':time_features,'lag':lag_features},'leakage_exclusions':excluded_leakage_features,'restock_policy':restock_policy,'rows_removed':int(rows_before-len(long)),'duplicate_rows_before_cleaning':duplicates_before,'missing_value_handling':'invalid dates/targets removed; categorical missing values mapped to Unknown; rolling warm-up values filled with 0; model pipeline imputes remaining features'}
(OUT/'visuelle_demand_preprocessing_summary.json').write_text(json.dumps(summary,indent=2,default=str),encoding='utf-8')
print(json.dumps(cfg,indent=2)); print('Saved corrected weekly artifacts to',OUT)

{
  "forecast_mode": "short_observation",
  "target_column": "demand",
  "time_column": "time",
  "time_unit": "week",
  "identifier_columns": [
    "external_code",
    "retail"
  ],
  "numerical_features": [
    "year",
    "month",
    "calendar_week",
    "day_of_week",
    "demand_lag_1",
    "demand_lag_3",
    "demand_rolling_mean_3",
    "demand_rolling_std_3",
    "restock_past_1",
    "price_past_1",
    "discount_past_1"
  ],
  "categorical_features": [
    "season",
    "season_from_time",
    "category",
    "color",
    "fabric",
    "shop_label"
  ],
  "time_features": [
    "year",
    "month",
    "calendar_week",
    "day_of_week",
    "season_from_time"
  ],
  "lag_features": [
    "demand_lag_1",
    "demand_lag_3",
    "demand_rolling_mean_3",
    "demand_rolling_std_3",
    "restock_past_1",
    "price_past_1",
    "discount_past_1"
  ],
  "feature_columns": [
    "season",
    "season_from_time",
    "category",
    "color",
    "fabric",
    "shop_label",
    "y

## Conclusion

This notebook emits weekly, disk-based splits and a feature configuration for `FORECAST_MODE`. `short_observation` may use only shifted demand/rolling and past observed restock/price/discount. `cold_start` excludes own demand/restock history and uses only product/shop categorical attributes, time features, and release-time product-specific price/discount when exact keys exist. Raw product IDs remain identifiers, not predictive numeric features. All lag and rolling calculations shift first; future/post-launch aggregate stock/restock is excluded.